# Why We Migrated from React to Next.js for the BAN Project

For the BAN project, we initially started with a standard React setup. However, as the project requirements became clearer, we realized that **Next.js** would provide several critical advantages that are necessary for our project’s success:

1. **Better Performance & SEO**
   - The BAN project involves dashboards, parent portals, and real-time patient data.
   - Server-Side Rendering (SSR) in Next.js ensures faster page loads and better SEO for public-facing pages.

    [Thats probably the least concern for now but yeah maybe have to defend in future you know]

2. **App Router & Structured Routing**
   - Next.js App Router allows us to organize pages and layouts efficiently.
   - Nested layouts help maintain consistent navigation, headers, and footers across different user roles (parents, staff).

3. **Scalability & Maintainability**
   - With Next.js, the project can grow without becoming messy.
   - Using the `src/` directory and App Router keeps components, hooks, utilities, and styles well-organized.

4. **Built-in Optimizations**
   - Automatic code splitting, image optimization, and performance enhancements come out-of-the-box.
   - This is especially important for dashboards that handle multiple real-time data streams.

5. **Future-Proof & Industry-Standard**
   - Next.js aligns with modern React best practices and provides a strong foundation for future updates.
   - Server Components and client/server separation help us write cleaner and more maintainable code.

### Summary
Switching to Next.js was **necessary** to ensure the BAN project is:
- Fast and optimized
- Structured and maintainable
- Ready for production
- Flexible enough to handle multiple user roles and real-time data



and for backend probably gonna use fast api?? for now that sound best to me whatever.



# **BAN Project: Complete stack**

## **1️⃣ Hardware: ESP32 Sensors**

* ESP32 sensors collect **real-time vitals**:

  * Heart Rate
  * SpO₂ (oxygen saturation)
  * Blood Pressure

* **ESP32 processing**:

  1. Reads raw sensor signals.
  2. Converts signals to **numeric values** (e.g., voltage → heart rate).
  3. Packages vitals into **JSON messages**:

     ```json
     {
       "heartRate": 78,
       "SpO2": 95,
       "BP": "120/80"
     }
     ```
  4. **Publishes the message** to MQTT broker using the **broker’s IP address**.

✅ **Why ESP32:** low-cost, Wi-Fi enabled, ideal for IoT and real-time sensor applications.

---

## **2️⃣ Communication Protocol: MQTT**

* **MQTT** is a **protocol for sending messages** between devices.
* **MQTT Broker** is the **software post office**:

  * Receives messages from sensors
  * Delivers them to the backend (FastAPI) immediately

**Why MQTT:**

* Lightweight → good for frequent, small messages from sensors
* Real-time delivery → backend receives data instantly
* Scalable → can handle multiple sensors at the same time
* Reliable → supports message delivery even if backend is temporarily down

---

## **3️⃣ Backend: FastAPI (Python)**

* FastAPI receives **messages from MQTT broker**.
* Handles:

  1. **Preprocessing** → cleans noisy or invalid data
  2. **Alert calculations** → triggers yellow/red alerts if thresholds exceeded
  3. **Predictive analysis** → optional ML models or trends
  4. **Stores data** → in a database for history and analytics
  5. **Exposes APIs** → frontend fetches processed data

**Why FastAPI:**

* Python is excellent for **data processing and predictive analytics**
* FastAPI is **async** → can handle real-time data efficiently
* Easy integration with MQTT and databases

---

## **4️⃣ Database**

* Stores **historical vitals, alerts, and predictions**.
* Recommended free options:

  * **PostgreSQL** → structured, reliable, supports complex queries
  * **SQLite** → simple file-based DB for testing

  also the data coming from sensors will be highly structure so using postgre sounds good to me so far i explore the dbs, its relational and we need taht guess so yewah

**Why a DB:**

* Keeps historical data for analysis
* Provides data for dashboards and alerts
* Backend queries DB to serve frontend

---

## **5️⃣ Frontend: Next.js (JavaScript)**

* **Next.js** fetches data from FastAPI APIs.
* Shows:

  * Real-time vitals
  * Alerts (yellow/red)
  * Parent portal simplified view

**Why Next.js:**

* React-based → fast development
* Supports **server-side rendering** and **dynamic routing**
* Works well with REST APIs from FastAPI

---

## **6️⃣ Complete Flow**

1. **ESP32 Sensor** → measures vitals → publishes to **MQTT broker**
2. **MQTT Broker** → receives messages → delivers to **FastAPI backend**
3. **FastAPI Backend** → preprocesses, calculates alerts/predictions, stores in DB, exposes API
4. **Database** → stores historical data for analysis
5. **Next.js Frontend** → fetches API → displays live dashboard & alerts

**Text Diagram:**

```
[ESP32 Sensors] 
      |
      v
[MQTT Broker (software)] 
      |
      v
[FastAPI Backend] --> [Database]
      |
      v
[Next.js Frontend Dashboard]
```

---

## **7️⃣ Code Examples**

### **ESP32 Publish Example**

```cpp
#include <WiFi.h>
#include <PubSubClient.h>

const char* ssid = "YOUR_WIFI";
const char* password = "YOUR_PASSWORD";
const char* mqtt_server = "BROKER_IP"; // MQTT broker IP

WiFiClient espClient;
PubSubClient client(espClient);

void setup() {
  Serial.begin(115200);
  WiFi.begin(ssid, password);
  while (WiFi.status() != WL_CONNECTED) delay(500);
  client.setServer(mqtt_server, 1883);
}

void loop() {
  if (!client.connected()) {
    while (!client.connected()) {
      if (client.connect("ESP32Client")) Serial.println("Connected to MQTT!");
      else delay(1000);
    }
  }
  int heartRate = random(70, 100);
  int SpO2 = random(90, 100);
  String payload = "{\"heartRate\": " + String(heartRate) + ", \"SpO2\": " + String(SpO2) + "}";
  client.publish("patient/123/vitals", payload.c_str());
  delay(2000);
}
```

### **FastAPI Subscribe Example**

```python
import json
from fastapi import FastAPI
import paho.mqtt.client as mqtt

app = FastAPI()
latest_vitals = {}

def on_connect(client, userdata, flags, rc):
    print("Connected to MQTT Broker")
    client.subscribe("patient/+/vitals")

def on_message(client, userdata, msg):
    global latest_vitals
    payload = json.loads(msg.payload.decode())
    patient_id = msg.topic.split("/")[1]
    latest_vitals[patient_id] = payload
    print(f"Patient {patient_id}: {payload}")

mqtt_client = mqtt.Client()
mqtt_client.on_connect = on_connect
mqtt_client.on_message = on_message
mqtt_client.connect("localhost", 1883)
mqtt_client.loop_start()

@app.get("/patients/{patient_id}/vitals")
def get_patient_vitals(patient_id: str):
    return latest_vitals.get(patient_id, {})
```

### **Next.js Frontend Fetch Example**

```js
import { useEffect, useState } from "react";

export default function Dashboard() {
  const [vitals, setVitals] = useState({});

  useEffect(() => {
    const interval = setInterval(async () => {
      const res = await fetch("http://localhost:8000/patients/123/vitals");
      const data = await res.json();
      setVitals(data);
    }, 2000);
    return () => clearInterval(interval);
  }, []);

  return (
    <div>
      <h1>Patient 123 Dashboard</h1>
      <p>Heart Rate: {vitals.heartRate || "-"}</p>
      <p>SpO2: {vitals.SpO2 || "-"}</p>
    </div>
  );
}
```

---

✅ **Summary of Choices**

| Component | Choice              | Why                                                                  |
| --------- | ------------------- | -------------------------------------------------------------------- |
| Sensor    | ESP32               | Low-cost, Wi-Fi, ideal for real-time vitals                          |
| Protocol  | MQTT                | Lightweight, real-time, reliable, scalable                           |
| Backend   | FastAPI             | Python for preprocessing & predictions, async, easy MQTT integration |
| Database  | PostgreSQL / SQLite | Stores historical vitals & alerts                                    |
| Frontend  | Next.js             | React-based, fast development, fetch API easily                      |

---

